# Phishing Website Detection Using Machine Learning

**Project:** Phishing Website Detection  
**Domain:** Cyber Security / Machine Learning

## Objective
Train a machine-learning model to classify website records as **legitimate** or **phishing** using the UCI Phishing Websites dataset. This notebook uses pre-extracted website security features rather than visiting websites or interacting with suspicious links.

## Dataset
UCI Machine Learning Repository — Phishing Websites:  
https://archive.ics.uci.edu/dataset/327/phishing+websites

Download the dataset from the UCI page and place the data file in the same folder as this notebook. The loader below supports common CSV/TXT exports. If the downloaded file uses a different filename or delimiter, update `DATA_PATH` and the loading section accordingly.

> Educational demonstration only. Model predictions can be wrong and should not be used as the sole basis for security decisions.


In [ ]:
# If needed, install dependencies in your notebook environment:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

RANDOM_STATE = 42
DATA_PATH = Path("phishing.csv")  # Change this to your downloaded dataset filename


## 1. Load the dataset

The UCI dataset may be supplied as a text file with a header or as a CSV export. The following cell tries common formats. Check the displayed columns and target labels after loading.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Download the UCI Phishing Websites dataset and place the data file here, "
        "then update DATA_PATH to match its filename."
    )

# Try comma-separated format first; fall back to whitespace-separated format.
try:
    df = pd.read_csv(DATA_PATH)
    if df.shape[1] == 1:
        df = pd.read_csv(DATA_PATH, sep=r"\\s+", engine="python")
except Exception:
    df = pd.read_csv(DATA_PATH, sep=r"\\s+", engine="python")

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:", list(df.columns))
print("\nMissing values:", int(df.isna().sum().sum()))


## 2. Prepare features and target

The UCI Phishing Websites dataset commonly encodes its target as `-1` (phishing), `1` (legitimate), and sometimes `0` for suspicious/unknown. We map `-1` to **phishing (1)** and `1` to **legitimate (0)**. The code also supports a target column named `Result` or a final-column target. Review your dataset's documentation if the target encoding differs.

In [ ]:
# Normalize column names for easier target detection.
df.columns = [str(c).strip() for c in df.columns]

target_candidates = [c for c in df.columns if c.lower() in {"result", "label", "target", "class"}]
target_col = target_candidates[0] if target_candidates else df.columns[-1]

data = df.copy()
y_raw = data.pop(target_col)

# Convert values to numeric when possible.
y_numeric = pd.to_numeric(y_raw, errors="coerce")
valid = y_numeric.notna()
data = data.loc[valid].copy()
y_numeric = y_numeric.loc[valid]

unique_labels = set(y_numeric.unique().tolist())
if unique_labels.issubset({-1, 1}):
    # UCI's usual encoding: -1 = phishing, 1 = legitimate
    y = (y_numeric == -1).astype(int)
elif unique_labels.issubset({0, 1}):
    # Assumption for binary exports: 1 = phishing, 0 = legitimate
    y = y_numeric.astype(int)
else:
    raise ValueError(
        f"Unexpected target labels: {sorted(unique_labels)}. "
        "Please update target mapping based on the dataset documentation."
    )

X = data.apply(pd.to_numeric, errors="coerce")
# Drop columns that contain no numeric data at all.
X = X.dropna(axis=1, how="all")

print("Target column:", target_col)
print("Feature matrix:", X.shape)
print("Label counts (0=legitimate, 1=phishing):")
print(y.value_counts().sort_index())


## 3. Split the data and train the model

A stratified split keeps the class proportions similar in training and test sets. The model uses a random forest classifier and median imputation for any missing feature values.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)


## 4. Evaluate performance

Accuracy alone can hide mistakes. Precision, recall, F1-score, and the confusion matrix provide a more complete view. In phishing detection, false negatives (phishing sites classified as legitimate) are especially important to monitor.

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions, zero_division=0),
    "Recall": recall_score(y_test, predictions, zero_division=0),
    "F1-score": f1_score(y_test, predictions, zero_division=0),
}
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, predictions,
    target_names=["Legitimate", "Phishing"],
    zero_division=0
))

cm = confusion_matrix(y_test, predictions, labels=[0, 1])
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate", "Phishing"]
).plot(values_format="d")
plt.title("Confusion Matrix — Phishing Website Detection")
plt.tight_layout()
plt.show()


## 5. Inspect important features

Feature importance can help explain which supplied features influenced the random forest. Importance does not prove causation and should be interpreted alongside domain knowledge.

In [ ]:
classifier = model.named_steps["classifier"]
importance = pd.Series(classifier.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False).head(15)

importance.sort_values().plot(kind="barh", figsize=(9, 6))
plt.xlabel("Feature importance")
plt.title("Top 15 Features")
plt.tight_layout()
plt.show()

display(importance.rename("importance").to_frame())


## 6. Save the trained model

The saved pipeline includes both missing-value handling and the classifier. Keep the model file with the project if you want to reuse it later.

In [ ]:
model_path = Path("phishing_detection_model.joblib")
joblib.dump(model, model_path)
print(f"Saved model to: {model_path.resolve()}")


## 7. Try a prediction on a feature record

Provide feature values using the same columns and feature definitions as the training dataset. This example uses a row from the test set so it is guaranteed to have the expected feature structure. It is not a substitute for evaluating unseen, real-world URLs.

In [ ]:
sample = X_test.iloc[[0]]
predicted_class = int(model.predict(sample)[0])
label = "PHISHING" if predicted_class == 1 else "LEGITIMATE"

print("Predicted class:", label)
print("Actual class:", "PHISHING" if int(y_test.iloc[0]) == 1 else "LEGITIMATE")
display(sample)


## Conclusion

This project demonstrates a reproducible machine-learning workflow for phishing website classification: loading a public dataset, preparing labels, training a model, evaluating performance, examining feature importance, and saving the pipeline.

### Limitations
- Dataset performance may not generalize to new phishing campaigns.
- The model uses dataset features; it does not inspect a live URL.
- Feature definitions and target-label encoding must match the dataset version.
- Always combine automated detection with browser protections, security tools, and human review.
